# 第11章 事务与并发控制

## 📌 学习目标

通过本章学习，你将掌握：
- 什么是事务
- ACID 四大特性
- START TRANSACTION / COMMIT / ROLLBACK
- 事务的隔离级别
- 锁的概念

---

In [ ]:
import os
import pymysql
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": "shop_db",
    "charset": "utf8mb4",
}

def run_query(sql, params=None):
    conn = pymysql.connect(**DB_CONFIG)
    try:
        return pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()

## 1. 什么是事务？

**事务（Transaction）** 是一组操作，要么**全部成功**，要么**全部失败**。

### 经典例子：银行转账

```
张三给李四转账 100 元：

步骤1: 张三账户 -100
步骤2: 李四账户 +100

如果步骤1成功，步骤2失败：
❌ 钱消失了！

使用事务：
✅ 两步都成功 → 提交（COMMIT）
❌ 任一步失败 → 回滚（ROLLBACK）
```

## 2. ACID 四大特性

| 特性 | 英文 | 说明 |
|------|------|------|
| **原子性** | Atomicity | 事务中的操作要么全部完成，要么全部不执行 |
| **一致性** | Consistency | 事务前后数据库保持一致状态 |
| **隔离性** | Isolation | 并发事务之间互不干扰 |
| **持久性** | Durability | 事务提交后，数据永久保存 |

## 3. 事务控制语句

```sql
START TRANSACTION;  -- 或 BEGIN;
-- 执行 SQL 操作
COMMIT;             -- 提交事务（所有操作生效）
-- 或
ROLLBACK;           -- 回滚事务（所有操作撤销）
```

In [ ]:
# 创建一个简单的账户表用于演示
execute_sql("DROP TABLE IF EXISTS accounts")
execute_sql("""
    CREATE TABLE accounts (
        id      INT PRIMARY KEY,
        name    VARCHAR(50),
        balance DECIMAL(10, 2)
    )
""")

execute_sql("INSERT INTO accounts VALUES (1, '张三', 1000.00)")
execute_sql("INSERT INTO accounts VALUES (2, '李四', 500.00)")
run_query("SELECT * FROM accounts")

### 3.1 COMMIT — 正常提交

In [ ]:
# 模拟转账：张三 → 李四 转 200 元
conn = pymysql.connect(**DB_CONFIG)
try:
    cursor = conn.cursor()
    
    # 开启事务
    cursor.execute("START TRANSACTION")
    
    # 步骤1：张三扣款
    cursor.execute("UPDATE accounts SET balance = balance - 200 WHERE id = 1")
    
    # 步骤2：李四收款
    cursor.execute("UPDATE accounts SET balance = balance + 200 WHERE id = 2")
    
    # 提交事务
    conn.commit()
    print("转账成功！")
finally:
    conn.close()

run_query("SELECT * FROM accounts")

### 3.2 ROLLBACK — 回滚事务

In [ ]:
# 模拟转账失败并回滚
conn = pymysql.connect(**DB_CONFIG)
try:
    cursor = conn.cursor()
    
    cursor.execute("START TRANSACTION")
    
    # 步骤1：张三扣款
    cursor.execute("UPDATE accounts SET balance = balance - 200 WHERE id = 1")
    
    # 模拟错误：李四ID错误
    cursor.execute("UPDATE accounts SET balance = balance + 200 WHERE id = 999")
    
    # 检查：如果没有更新到记录，说明出错了
    if cursor.rowcount == 0:
        print("转账失败，回滚事务")
        conn.rollback()
    else:
        conn.commit()
        print("转账成功")
finally:
    conn.close()

# 验证：余额应该没变
run_query("SELECT * FROM accounts")

### 3.3 SAVEPOINT — 保存点

在事务中设置保存点，可以回滚到指定位置，而不是整个事务。

```sql
SAVEPOINT 保存点名;
ROLLBACK TO 保存点名;
```

In [ ]:
# 使用保存点
conn = pymysql.connect(**DB_CONFIG)
try:
    cursor = conn.cursor()
    cursor.execute("START TRANSACTION")
    
    # 第1步操作
    cursor.execute("UPDATE accounts SET balance = balance - 100 WHERE id = 1")
    cursor.execute("SAVEPOINT step1")
    
    # 第2步操作（出错）
    cursor.execute("UPDATE accounts SET balance = balance + 100 WHERE id = 999")
    
    # 回滚到保存点
    cursor.execute("ROLLBACK TO step1")
    
    # 第2步重新执行
    cursor.execute("UPDATE accounts SET balance = balance + 100 WHERE id = 2")
    
    conn.commit()
    print("使用保存点，部分回滚成功")
finally:
    conn.close()

run_query("SELECT * FROM accounts")

## 4. 隔离级别

当多个事务同时执行时，可能出现以下问题：

| 问题 | 说明 | 例子 |
|------|------|------|
| **脏读** | 读到未提交的数据 | 读到别人还没确认的修改 |
| **不可重复读** | 同一事务中两次读取结果不同 | 两次查余额不一样 |
| **幻读** | 同一事务中两次查询行数不同 | 第一次查5条，第二次查6条 |

### MySQL 隔离级别

| 隔离级别 | 脏读 | 不可重复读 | 幻读 |
|----------|------|------------|------|
| READ UNCOMMITTED | ✅ 可能 | ✅ 可能 | ✅ 可能 |
| READ COMMITTED | ❌ 不会 | ✅ 可能 | ✅ 可能 |
| **REPEATABLE READ**（默认） | ❌ 不会 | ❌ 不会 | ✅ 可能 |
| SERIALIZABLE | ❌ 不会 | ❌ 不会 | ❌ 不会 |

```sql
-- 查看当前隔离级别
SELECT @@transaction_isolation;

-- 设置隔离级别
SET SESSION TRANSACTION ISOLATION LEVEL READ COMMITTED;
```

In [ ]:
# 查看当前隔离级别
run_query("SELECT @@transaction_isolation AS 隔离级别")

## 5. 锁

MySQL 使用锁来保证并发安全。

### 锁的类型

| 锁类型 | 说明 |
|--------|------|
| **共享锁（S锁）** | 读锁，多个事务可以同时读 |
| **排他锁（X锁）** | 写锁，只有一个事务可以写 |
| **表锁** | 锁定整张表 |
| **行锁** | 锁定某一行（InnoDB 支持） |

```sql
SELECT ... FOR SHARE;      -- 共享锁
SELECT ... FOR UPDATE;     -- 排他锁
```

In [ ]:
# 清理演示表
execute_sql("DROP TABLE IF EXISTS accounts")
print("演示表已清理")

## 📝 练习题

### 题目1（简单）
创建一个 `wallet` 表（id, name, balance），插入两条数据，然后使用事务将 id=1 的余额减少 100，id=2 的余额增加 100，最后提交。

### 题目2（中等）
编写一个事务：尝试将 id=1 的余额减少 10000（超过余额），如果余额不足则回滚。

### 题目3（中等）
查看当前数据库的隔离级别。

---

## ✅ 参考答案

In [ ]:
# 题目1：创建表并使用事务
execute_sql("DROP TABLE IF EXISTS wallet")
execute_sql("""
    CREATE TABLE wallet (
        id      INT PRIMARY KEY,
        name    VARCHAR(50),
        balance DECIMAL(10, 2)
    )
""")
execute_sql("INSERT INTO wallet VALUES (1, '张三', 500.00)")
execute_sql("INSERT INTO wallet VALUES (2, '李四', 300.00)")

# 使用事务转账
conn = pymysql.connect(**DB_CONFIG)
try:
    cursor = conn.cursor()
    cursor.execute("START TRANSACTION")
    cursor.execute("UPDATE wallet SET balance = balance - 100 WHERE id = 1")
    cursor.execute("UPDATE wallet SET balance = balance + 100 WHERE id = 2")
    conn.commit()
    print("转账成功")
finally:
    conn.close()

run_query("SELECT * FROM wallet")

In [ ]:
# 题目2：余额不足时回滚
conn = pymysql.connect(**DB_CONFIG)
try:
    cursor = conn.cursor()
    cursor.execute("START TRANSACTION")
    
    # 检查余额
    cursor.execute("SELECT balance FROM wallet WHERE id = 1")
    balance = cursor.fetchone()[0]
    
    if balance < 10000:
        print(f"余额不足（当前: {balance}），回滚事务")
        conn.rollback()
    else:
        cursor.execute("UPDATE wallet SET balance = balance - 10000 WHERE id = 1")
        conn.commit()
        print("扣款成功")
finally:
    conn.close()

run_query("SELECT * FROM wallet")

In [ ]:
# 题目3：查看隔离级别
run_query("SELECT @@transaction_isolation AS 当前隔离级别")

In [ ]:
# 清理
execute_sql("DROP TABLE IF EXISTS wallet")

---

**下一章：[12_综合实战](12_综合实战.ipynb)**